In [3]:
import pandas as pd
from pathlib import Path

# ============================================================================
# Project paths
# ============================================================================
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
TABLE_DIR = PROJECT_ROOT / 'results' / 'table'

print("Project paths ready")

print("=" * 80)
print("Appendix Case Selection: Diagnostic for Supplementary Material (S1-S4)")
print("Selects 4 representative cases, one per industry, diversified by grade")
print("=" * 80)

# ============================================================================
# 1. Data Loading
# ============================================================================
agent1 = pd.read_csv(DATA_DIR / 'agent1_interpretation_results_4industry.csv')
agent2 = pd.read_csv(DATA_DIR / 'agent2_consulting_reports_4industry.csv')
agent3 = pd.read_csv(DATA_DIR / 'agent3_ensemble_results_4industry.csv')

TARGET_SIC_CODES = ['G46', 'G47', 'L68', 'F42']
SIC_LABELS = {
    'G46': 'Wholesale Trade',
    'G47': 'Retail Trade',
    'L68': 'Real Estate',
    'F42': 'Construction',
}

print(f"\nLoaded Agent #1: {len(agent1)} records")
print(f"Loaded Agent #2: {len(agent2)} records")
print(f"Loaded Agent #3: {len(agent3)} records")

# ============================================================================
# 2. Check grade availability per industry — needed to know which industries
#    can supply a Reject case, and which grades exist at all in each sector.
# ============================================================================
print("\n" + "=" * 80)
print("Grade availability by industry")
print("=" * 80)
grade_table = pd.crosstab(agent3['SIC_CD_3'], agent3['decision'])
print(grade_table)

# ============================================================================
# 3. Greedy assignment: prioritize covering a Reject case first (in whichever
#    industry has one, preferring the industry with the FEWEST Reject cases,
#    since that makes the selected case more representative of a genuine
#    edge case rather than a common outcome for that sector). Remaining
#    industries are then assigned diversified grades (Pass / Conditional
#    Pass), cycling through a preference order and falling back to whatever
#    grade actually exists for that industry if the preferred one is absent.
# ============================================================================
print("\n" + "=" * 80)
print("Selecting one representative case per industry, maximizing grade diversity")
print("=" * 80)

reject_available = agent3[agent3['decision'] == 'Reject']['SIC_CD_3'].value_counts()
print(f"\nIndustries with at least one Reject case (case count):\n{reject_available}")

reject_sic = reject_available.index[-1] if len(reject_available) > 0 else None  # rarest first
print(f"\n-> Assigning Reject case to: {reject_sic}")

assignments = {}
if reject_sic:
    assignments[reject_sic] = 'Reject'

remaining_sics = [s for s in TARGET_SIC_CODES if s not in assignments]
remaining_grades_priority = ['Pass', 'Conditional Pass', 'Pass']

grade_idx = 0
for sic in remaining_sics:
    available_grades = agent3[agent3['SIC_CD_3'] == sic]['decision'].unique()
    assigned = False
    for _ in range(len(remaining_grades_priority)):
        candidate_grade = remaining_grades_priority[grade_idx % len(remaining_grades_priority)]
        grade_idx += 1
        if candidate_grade in available_grades:
            assignments[sic] = candidate_grade
            assigned = True
            break
    if not assigned:
        assignments[sic] = available_grades[0]  # fallback: whatever grade exists

print(f"\nFinal assignments: {assignments}")

# ============================================================================
# 4. Within each assigned (industry, grade) pair, select the highest-scoring
#    case as the representative example.
# ============================================================================
selected_cases = []
for sic, grade in assignments.items():
    subset = agent3[(agent3['SIC_CD_3'] == sic) & (agent3['decision'] == grade)]
    if subset.empty:
        print(f"[Warning] No case found for {sic}/{grade}")
        continue
    best_row = subset.loc[subset['final_score'].idxmax()]
    selected_cases.append({
        'SIC_CD_3': sic,
        'industry_label': SIC_LABELS[sic],
        'company_id': int(best_row['company_id']),
        'grade': grade,
        'final_score': best_row['final_score'],
    })

selected_df = pd.DataFrame(selected_cases)
print("\n" + "=" * 80)
print("SELECTED REPRESENTATIVE CASES (4 industries, diversified grades)")
print("=" * 80)
print(selected_df.to_string(index=False))

# ============================================================================
# 5. Print full detail for each selected case: Agent #1 probability shift and
#    feasibility label, Agent #3 per-dimension scores, and an Agent #2 report
#    excerpt — everything needed to draft Supplementary sections S1-S4.
# ============================================================================
for _, row in selected_df.iterrows():
    cid = row['company_id']
    print("\n" + "=" * 80)
    print(f"CASE: {cid} | {row['industry_label']} ({row['SIC_CD_3']}) | "
          f"Grade: {row['grade']} | Score: {row['final_score']}")
    print("=" * 80)

    a1_row = agent1[agent1['ID'] == cid]
    a2_row = agent2[agent2['company_id'] == cid]
    a3_row = agent3[agent3['company_id'] == cid]

    if not a1_row.empty:
        a1 = a1_row.iloc[0]
        print(f"\n[Agent #1] Bankruptcy probability: {a1['Original_Proba']:.1%} -> {a1['Target_Proba']:.1%}")
        print(f"[Agent #1] Feasibility assessment: {a1.get('feasibility_assessment', 'N/A')}")
        print(f"[Agent #1] Business meaning: {a1.get('business_meaning', 'N/A')}")

    if not a3_row.empty:
        a3 = a3_row.iloc[0]
        print(f"\n[Agent #3] CF Alignment: {a3['score_cf_alignment']} "
              f"(match_rate: {a3.get('cf_alignment_match_rate', 'N/A')}, "
              f"matched: {a3.get('cf_alignment_n_matched', 'N/A')}/{a3.get('cf_alignment_n_ground_truth', 'N/A')})")
        print(f"[Agent #3] Logic & Flow: {a3['score_logic_flow']} | Actionability: {a3['score_actionability']}")
        print(f"[Agent #3] Business Insight: {a3['score_business_insight']} | "
              f"Completeness: {a3['score_completeness']} | Clarity: {a3['score_clarity']}")

    if not a2_row.empty:
        print(f"\n[Agent #2 Report excerpt]")
        print(a2_row.iloc[0]['report_content'][:600])
        print("...")

# ============================================================================
# 6. Save the selection so it can be referenced when drafting
#    sections/06_supplementary.tex
# ============================================================================
output_path = TABLE_DIR / 'appendix_case_selection.csv'
selected_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"\nSelection saved: {output_path}")

Project paths ready
Appendix Case Selection: Diagnostic for Supplementary Material (S1-S4)
Selects 4 representative cases, one per industry, diversified by grade

Loaded Agent #1: 542 records
Loaded Agent #2: 542 records
Loaded Agent #3: 542 records

Grade availability by industry
decision  Conditional Pass  Pass  Reject
SIC_CD_3                                
F42                     47    22      27
G46                     91    44      53
G47                     71    42      25
L68                     71    25      24

Selecting one representative case per industry, maximizing grade diversity

Industries with at least one Reject case (case count):
SIC_CD_3
G46    53
F42    27
G47    25
L68    24
Name: count, dtype: int64

-> Assigning Reject case to: L68

Final assignments: {'L68': 'Reject', 'G46': 'Pass', 'G47': 'Conditional Pass', 'F42': 'Pass'}

SELECTED REPRESENTATIVE CASES (4 industries, diversified grades)
SIC_CD_3  industry_label  company_id            grade  final_score
   